# Analise de Jogos - Brasileirao 2025

Analise exploratoria comparando o desempenho dos principais clubes do Brasileirao 2025.
Os dados incluem gols, escanteios e cartoes amarelos dos ultimos 5 jogos de cada time.

**Objetivos:**
- Comparar estatisticas entre dois times selecionados
- Visualizar a evolucao do desempenho jogo a jogo
- Gerar um panorama geral do campeonato via heatmap

**Stack:** pandas, matplotlib, seaborn

## 1. Configuracao e Importacoes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

COR_A = '#1f77b4'
COR_B = '#ff7f0e'
TIMES = ['Flamengo', 'Palmeiras', 'Corinthians', 'Sao Paulo', 'Gremio', 'Fortaleza']

print('Bibliotecas carregadas!')

## 2. Carregamento dos Dados

In [ ]:
df = pd.read_csv('dados/partidas.csv', parse_dates=['data'])
print(f'Total de partidas: {len(df)}')
df.head(10)

## 3. Estatisticas Descritivas

In [ ]:
cols = ['gols_mandante','gols_visitante','escanteios_mandante',
        'escanteios_visitante','cartoes_amarelos_mandante','cartoes_amarelos_visitante']
df[cols].describe().round(2)

## 4. Funcao de Analise por Time

In [ ]:
def calcular_estatisticas(df, time):
    partidas = df[(df['time_mandante'] == time) | (df['time_visitante'] == time)].copy()
    partidas = partidas.sort_values('data').tail(5)
    gols, escanteios, cartoes, resultados = [], [], [], []
    for _, row in partidas.iterrows():
        mand = row['time_mandante'] == time
        gols.append(int(row['gols_mandante'] if mand else row['gols_visitante']))
        escanteios.append(int(row['escanteios_mandante'] if mand else row['escanteios_visitante']))
        cartoes.append(int(row['cartoes_amarelos_mandante'] if mand else row['cartoes_amarelos_visitante']))
        gf = row['gols_mandante'] if mand else row['gols_visitante']
        gc = row['gols_visitante'] if mand else row['gols_mandante']
        resultados.append('V' if gf > gc else ('D' if gf < gc else 'E'))
    return {
        'Gols por Jogo': gols, 'Media de Gols': round(sum(gols)/len(gols), 2),
        'Escanteios por Jogo': escanteios, 'Media de Escanteios': round(sum(escanteios)/len(escanteios), 2),
        'Cartoes por Jogo': cartoes, 'Media de Cartoes': round(sum(cartoes)/len(cartoes), 2),
        'Vitorias': resultados.count('V'), 'Empates': resultados.count('E'), 'Derrotas': resultados.count('D'),
    }

print('Funcao pronta!')

## 5. Selecao dos Times

> Altere **TIME_A** e **TIME_B** para comparar outros clubes

In [ ]:
TIME_A = 'Flamengo'
TIME_B = 'Palmeiras'

sa = calcular_estatisticas(df, TIME_A)
sb = calcular_estatisticas(df, TIME_B)

pd.DataFrame([sa, sb], index=[TIME_A, TIME_B])[[
    'Media de Gols','Media de Escanteios','Media de Cartoes','Vitorias','Empates','Derrotas'
]]

## 6. Grafico 1 - Barras Agrupadas por Jogo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(f'Estatisticas - {TIME_A} vs {TIME_B} (ultimos 5 jogos)', fontsize=14, fontweight='bold')
pares = [
    ('Gols por Jogo','Media de Gols','Gols'),
    ('Escanteios por Jogo','Media de Escanteios','Escanteios'),
    ('Cartoes por Jogo','Media de Cartoes','Cartoes Amarelos'),
]
x = range(1, 6)
for ax, (chave, media, ylabel) in zip(axes, pares):
    ax.bar([j-0.2 for j in x], sa[chave], width=0.4, color=COR_A, label=f'{TIME_A} (mu={sa[media]})')
    ax.bar([j+0.2 for j in x], sb[chave], width=0.4, color=COR_B, label=f'{TIME_B} (mu={sb[media]})')
    ax.axhline(sa[media], color=COR_A, linestyle='--', linewidth=1, alpha=0.7)
    ax.axhline(sb[media], color=COR_B, linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(ylabel); ax.set_xlabel('Jogo')
    ax.set_xticks(list(x)); ax.set_xticklabels([f'J{j}' for j in x])
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. Grafico 2 - Aproveitamento (Pizza)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Aproveitamento - ultimos 5 jogos', fontsize=14, fontweight='bold')
for ax, (stats, time) in zip(axes, [(sa, TIME_A), (sb, TIME_B)]):
    ax.pie([stats['Vitorias'], stats['Empates'], stats['Derrotas']],
           labels=['Vitorias','Empates','Derrotas'],
           colors=['#2ecc71','#f39c12','#e74c3c'],
           autopct='%1.0f%%', startangle=90, explode=[0.05,0,0])
    ax.set_title(time, fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Grafico 3 - Evolucao por Jogo (Linha)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'Evolucao - {TIME_A} vs {TIME_B}', fontsize=14, fontweight='bold')
pares = [('Gols por Jogo','Gols'),('Escanteios por Jogo','Escanteios'),('Cartoes por Jogo','Cartoes')]
x = range(1, 6)
for ax, (chave, ylabel) in zip(axes, pares):
    ax.plot(x, sa[chave], marker='o', color=COR_A, label=TIME_A, linewidth=2)
    ax.plot(x, sb[chave], marker='s', color=COR_B, label=TIME_B, linewidth=2)
    ax.fill_between(x, sa[chave], sb[chave], alpha=0.1, color='gray')
    ax.set_title(ylabel)
    ax.set_xticks(list(x)); ax.set_xticklabels([f'J{j}' for j in x])
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 9. Grafico 4 - Heatmap Geral de Todos os Times

In [ ]:
registros = []
for time in TIMES:
    s = calcular_estatisticas(df, time)
    registros.append({
        'Time': time,
        'Media Gols': s['Media de Gols'],
        'Media Escanteios': s['Media de Escanteios'],
        'Media Cartoes': s['Media de Cartoes'],
        'Aproveitamento %': round((s['Vitorias']*3 + s['Empates']) / 15 * 100, 1),
    })
tabela = pd.DataFrame(registros).set_index('Time')
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(tabela, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Desempenho Geral - Brasileirao 2025', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 10. Conclusoes

Com base nos ultimos 5 jogos analisados:

- **Gols:** Compare as medias para identificar o ataque mais eficiente
- **Escanteios:** Indicador de dominio territorial e pressao ofensiva
- **Cartoes amarelos:** Reflete o estilo de jogo (agressivo vs. tecnico)
- **Aproveitamento:** % de pontos conquistados sobre os disputados

> Para comparar outros times, altere **TIME_A** e **TIME_B** na celula 5 e reexecute.